# Titanic Survival Prediction

Predicts passenger survival using four tuned models compared against each other — the best performer is selected automatically for the final submission.

## Notebook Structure

| Cell | Description |
|------|-------------|
| 1 | Load `train.csv` and `test.csv` |
| 2 | Impute missing values, encode categorical features |
| 3 | Baseline Random Forest — 80/20 train/val split |
| 4 | Hyperparameter tuning — Random Forest (162 combinations, 5-fold CV) |
| 5 | Hyperparameter tuning — Logistic Regression (18 combinations, 5-fold CV) |
| 6 | Hyperparameter tuning — XGBoost (108 combinations, 5-fold CV) |
| 7 | Hyperparameter tuning — SVM (20 combinations, 5-fold CV) |
| 8 | Preprocess test data, predict with best model, save `submission.csv` |

## Kaggle Results

| Submission | Model | Params | Public Score |
|------------|-------|--------|--------------|
| 1 | SVM | `C=1, kernel=rbf, gamma=scale` | **0.78229** |

In [31]:
import pandas as pd

# Load train and test datasets from CSV files
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Inspect available feature columns
print(train_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [ ]:
# Fill missing Age values with the column mean to avoid dropping rows
train_data['Age'].fillna(train_data['Age'].mean(), inplace=True)
# Fill missing Embarked with the most common port
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)

# Encode Sex as binary: male=0, female=1
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
# Encode Embarked as ordinal: C=0, Q=1, S=2
train_data['Embarked'] = train_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

C:\Users\BenCliffe\AppData\Local\Temp\ipykernel_8828\1693718049.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data['Age'].fillna(train_data['Age'].mean(), inplace=True)
C:\Users\BenCliffe\AppData\Local\Temp\ipykernel_8828\1693718049.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behav

In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Features selected for training — excludes Name, Ticket, Cabin due to high cardinality or missing data
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = train_data[features]
y = train_data['Survived']

# Hold out 20% of training data for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest with 100 decision trees
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate accuracy on the held-out validation set
accuracy = model.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy}")

Validation Accuracy: 0.8100558659217877


In [34]:
from sklearn.model_selection import GridSearchCV

# Parameter grid to search over
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# 5-fold cross-validation over all parameter combinations; n_jobs=-1 uses all CPU cores
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters:    {grid_search.best_params_}")
print(f"Best CV accuracy:   {grid_search.best_score_:.4f}")

# Replace model with the best estimator found
model = grid_search.best_estimator_
tuned_accuracy = model.score(X_val, y_val)
print(f"Tuned val accuracy: {tuned_accuracy:.4f}  (baseline: {accuracy:.4f})")

Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best parameters:    {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best CV accuracy:   0.8258
Tuned val accuracy: 0.8101  (baseline: 0.8101)


In [35]:
from sklearn.linear_model import LogisticRegression

# Two grids handle solver/penalty compatibility:
# liblinear supports both l1 (sparse, zeros out weak features) and l2 (shrinks all weights)
# lbfgs supports l2 only but scales better to larger datasets
param_grid_lr = [
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs']}
]

# max_iter=1000 prevents convergence warnings on harder parameter combinations
grid_search_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_lr.best_params_}")
print(f"Best CV accuracy:   {grid_search_lr.best_score_:.4f}")

lr_model = grid_search_lr.best_estimator_
lr_accuracy = lr_model.score(X_val, y_val)

# Compare LR against the tuned Random Forest and keep the better model
print(f"\nLR val accuracy:    {lr_accuracy:.4f}")
print(f"RF val accuracy:    {tuned_accuracy:.4f}")

model = lr_model if lr_accuracy > tuned_accuracy else model
print(f"\nSelected model: {'Logistic Regression' if lr_accuracy > tuned_accuracy else 'Random Forest'}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best parameters:    {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}
Best CV accuracy:   0.8005

LR val accuracy:    0.7933
RF val accuracy:    0.8101

Selected model: Random Forest


c:\Users\BenCliffe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\BenCliffe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [36]:
# If not installed: pip install xgboost
from xgboost import XGBClassifier

# Key XGBoost parameters to tune:
# n_estimators   — number of boosting rounds (more = slower but potentially better)
# max_depth       — max tree depth; lower = simpler model, less overfitting
# learning_rate   — shrinks each tree's contribution; lower needs more n_estimators
# subsample       — fraction of rows sampled per tree; <1.0 adds randomness to reduce overfitting
# colsample_bytree — fraction of features sampled per tree; similar effect to subsample
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# eval_metric='logloss' suppresses XGBoost's default metric warning
grid_search_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_xgb.best_params_}")
print(f"Best CV accuracy:   {grid_search_xgb.best_score_:.4f}")

xgb_model = grid_search_xgb.best_estimator_
xgb_accuracy = xgb_model.score(X_val, y_val)

# Compare all three tuned models and set model to the overall winner
print(f"\nXGB val accuracy:   {xgb_accuracy:.4f}")
print(f"RF  val accuracy:   {tuned_accuracy:.4f}")
print(f"LR  val accuracy:   {lr_accuracy:.4f}")

scores = {'XGBoost': (xgb_accuracy, xgb_model), 'Random Forest': (tuned_accuracy, model), 'Logistic Regression': (lr_accuracy, lr_model)}
best_name, (best_score, best_model) = max(scores.items(), key=lambda x: x[1][0])
model = best_model
print(f"\nSelected model: {best_name} ({best_score:.4f})")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters:    {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Best CV accuracy:   0.8399

XGB val accuracy:   0.7989
RF  val accuracy:   0.8101
LR  val accuracy:   0.7933

Selected model: Random Forest (0.8101)


In [37]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# SVM is sensitive to feature scale so StandardScaler is applied inside a Pipeline
# This ensures scaling is fitted only on training folds during cross-validation, preventing data leakage
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

# Two grids: gamma only affects rbf/poly kernels, not linear
# C controls the margin penalty — higher C = less tolerance for misclassification
# gamma controls how far a single training example's influence reaches (rbf/poly only)
param_grid_svm = [
    {'svc__C': [0.1, 1, 10, 100], 'svc__kernel': ['linear']},
    {'svc__C': [0.1, 1, 10, 100], 'svc__kernel': ['rbf', 'poly'], 'svc__gamma': ['scale', 'auto']}
]

grid_search_svm = GridSearchCV(
    pipeline_svm,
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_svm.best_params_}")
print(f"Best CV accuracy:   {grid_search_svm.best_score_:.4f}")

svm_model = grid_search_svm.best_estimator_
svm_accuracy = svm_model.score(X_val, y_val)

# Compare SVM against the current best model (winner of RF / LR / XGB)
current_best_accuracy = model.score(X_val, y_val)
print(f"\nSVM val accuracy:   {svm_accuracy:.4f}")
print(f"Current best:       {current_best_accuracy:.4f}")

if svm_accuracy > current_best_accuracy:
    model = svm_model
    print("\nSelected model: SVM")
else:
    print("\nPrevious best model retained")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:    {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best CV accuracy:   0.8244

SVM val accuracy:   0.8156
Current best:       0.8101

Selected model: SVM


In [38]:
# Apply the same preprocessing to test data before predicting
test_data['Age'].fillna(test_data['Age'].mean(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].mean(), inplace=True)
# Fill Embarked before mapping — NaN rows would silently become NaN after map otherwise
test_data['Embarked'].fillna(test_data['Embarked'].mode()[0], inplace=True)
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

X_test = test_data[features]

# Safety fill — catches any residual NaN that would cause SVM / scaled models to fail
X_test = X_test.fillna(X_test.median())

# Generate survival predictions for the test set
predictions = model.predict(X_test)

# Save predictions in Kaggle submission format (PassengerId + Survived)
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Submission file created!")

Submission file created!


C:\Users\BenCliffe\AppData\Local\Temp\ipykernel_8828\3169564710.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_data['Age'].fillna(test_data['Age'].mean(), inplace=True)
C:\Users\BenCliffe\AppData\Local\Temp\ipykernel_8828\3169564710.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves